## Route frequency hourly

In [0]:
%sql
CREATE OR REPLACE TABLE calgary_transit.gold.route_frequency_hourly AS
WITH trips_f AS (
  SELECT trip_id, route_id, direction_id
  FROM calgary_transit.bronze.gtfs_trips
  WHERE _ingest_date = '2026-02-23'
),
times AS (
  SELECT
    t.route_id,
    t.direction_id,
    -- take first departure_time per trip (at stop_sequence=1) as “trip start”
    CAST(SUBSTRING(st.departure_time, 1, 2) AS INT) AS hour_of_day
  FROM trips_f t
  JOIN calgary_transit.bronze.gtfs_stop_times st
    ON st.trip_id = t.trip_id
   AND st._ingest_date = '2026-02-23'
  WHERE st.stop_sequence = 1
)
SELECT
  route_id,
  direction_id,
  hour_of_day,
  COUNT(*) AS trip_count
FROM times
GROUP BY route_id, direction_id, hour_of_day;

In [0]:
%sql
select * from calgary_transit.gold.route_frequency_hourly;